<a href="https://colab.research.google.com/github/KnockriInc/Scale-Validation/blob/main/Scale_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook provides sample prompts to generate usable code for the purpose of providing inital evidence regarding the reliability and validity of a new measure. This example notebook is based on the development of a multi-dimensional scale to measure the contruct of AI-Readiness. However, the notebook is desgined to be easily adapted for different single dimension or multiple dimensional constructs.

The notebook is broken down into 4 phases that are as follows:

Phase 1 - Item Generation, Reduction, and Refinement

Phase 2 - Analysis of Basic Psychometric Properties

Phase 3 - Convergent, Discriminant, and Incremental Validity

Phase 4 - Criterion-Related Validity

For each phase, a prompt is provided that can be used to generate the code necessary to run each step in the phase. Multiple prompts are used when neccessary. In addition, code generated through the use of LLMs is also provided.

In [ ]:
!pip install pandas numpy scipy factor_analyzer semopy openpyxl

Phase 1 -  Item Generation, Reduction, and Refinement

Phase 2 - Analysis of Basic Psychometric Properties

Study 1: Exploratory Factor Analysis

In [ ]:
# =========================
# Exploratory Factor Analysis
# Principal Axis Factoring
# Oblique Rotation
# Forced 6-factor solution
# =========================

import pandas as pd
import numpy as np
from factor_analyzer import FactorAnalyzer
from scipy.stats import zscore

# -------------------------
# USER SETTINGS
# -------------------------

DATA_FILE = "EFA_ai_readiness_data.csv"
N_FACTORS = 6
LOADING_CUTOFF = 0.30
CROSS_LOADING_DIFF = 0.15
ROTATION = "promax"   # oblique rotation
METHOD = "principal"  # closest common factor extraction available in factor_analyzer

# Optional:
# If you want to specify item columns manually, list them here.
# Otherwise, all numeric columns will be used.
ITEM_COLUMNS = None

# Optional:
# Provide expected theoretical mapping if you want automated flagging
# of items that load on a factor inconsistent with their a priori factor.
# Example:
# A_PRIORI_MAP = {
#     "item1": 1,
#     "item2": 1,
#     "item3": 2,
# }
A_PRIORI_MAP = {}

# -------------------------
# HELPER FUNCTIONS
# -------------------------

def cronbach_alpha(df_items):
    """
    Compute Cronbach's alpha for a set of items.
    """
    df_items = df_items.dropna(axis=0, how="any")
    k = df_items.shape[1]

    if k < 2:
        return np.nan

    item_variances = df_items.var(axis=0, ddof=1)
    total_score = df_items.sum(axis=1)
    total_variance = total_score.var(ddof=1)

    if total_variance == 0:
        return np.nan

    alpha = (k / (k - 1)) * (1 - item_variances.sum() / total_variance)
    return alpha


def classify_item(loadings_row, cutoff=0.30, cross_loading_diff=0.15):
    """
    Classify item retention based on:
    - highest loading >= cutoff
    - cross-loading if second highest loading is too close
    """
    abs_loadings = loadings_row.abs().sort_values(ascending=False)
    primary_factor = abs_loadings.index[0]
    primary_loading = abs_loadings.iloc[0]
    secondary_loading = abs_loadings.iloc[1] if len(abs_loadings) > 1 else 0

    retain = primary_loading >= cutoff
    cross_loading = (secondary_loading >= cutoff) and ((primary_loading - secondary_loading) < cross_loading_diff)

    return {
        "primary_factor": primary_factor,
        "primary_loading": primary_loading,
        "secondary_loading": secondary_loading,
        "retain": retain and not cross_loading,
        "cross_loading": cross_loading
    }


def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.3f}".replace("0.", ".")


def apa_alpha(alpha):
    if pd.isna(alpha):
        return "NA"
    return f"{alpha:.2f}"


# -------------------------
# LOAD DATA
# -------------------------

df = pd.read_csv(DATA_FILE)

if ITEM_COLUMNS is None:
    item_df = df.select_dtypes(include=[np.number]).copy()
else:
    item_df = df[ITEM_COLUMNS].copy()

# Drop rows missing all item responses
item_df = item_df.dropna(how="all")

# Optional: drop zero-variance items
zero_var_items = item_df.columns[item_df.nunique(dropna=True) <= 1].tolist()
if zero_var_items:
    print("Dropping zero-variance items:", zero_var_items)
    item_df = item_df.drop(columns=zero_var_items)

# -------------------------
# FIT EFA
# -------------------------
# NOTE:
# factor_analyzer does not literally label extraction as "principal axis factoring"
# in the same way SPSS does.
# In practice, researchers often use this package for EFA with common-factor methods.
# If you require exact software replication, compare against SPSS/R psych::fa(fm='pa').

fa = FactorAnalyzer(
    n_factors=N_FACTORS,
    rotation=ROTATION,
    method=METHOD
)

fa.fit(item_df)

# Loadings
loading_matrix = pd.DataFrame(
    fa.loadings_,
    index=item_df.columns,
    columns=[f"Factor{i+1}" for i in range(N_FACTORS)]
)

# Factor correlation matrix (available for oblique solutions)
phi = None
try:
    phi = pd.DataFrame(
        fa.phi_,
        index=[f"Factor{i+1}" for i in range(N_FACTORS)],
        columns=[f"Factor{i+1}" for i in range(N_FACTORS)]
    )
except Exception:
    pass

# Variance explained
variance_info = fa.get_factor_variance()
variance_df = pd.DataFrame({
    "SS_Loadings": variance_info[0],
    "Proportion_Var": variance_info[1],
    "Cumulative_Var": variance_info[2]
}, index=[f"Factor{i+1}" for i in range(N_FACTORS)])

# -------------------------
# ITEM RETENTION RULES
# -------------------------

retention_rows = []

for item in loading_matrix.index:
    result = classify_item(
        loading_matrix.loc[item],
        cutoff=LOADING_CUTOFF,
        cross_loading_diff=CROSS_LOADING_DIFF
    )

    theoretical_inconsistency = False
    expected_factor = A_PRIORI_MAP.get(item, None)

    if expected_factor is not None:
        assigned_factor_num = int(result["primary_factor"].replace("Factor", ""))
        if assigned_factor_num != expected_factor:
            theoretical_inconsistency = True

    retain_final = result["retain"] and not theoretical_inconsistency

    retention_rows.append({
        "Item": item,
        "Assigned_Factor": result["primary_factor"],
        "Primary_Loading": round(result["primary_loading"], 3),
        "Secondary_Loading": round(result["secondary_loading"], 3),
        "Cross_Loading": result["cross_loading"],
        "Theoretically_Inconsistent": theoretical_inconsistency,
        "Retain": retain_final
    })

retention_df = pd.DataFrame(retention_rows)

# Retained items only
retained_items_df = retention_df[retention_df["Retain"]].copy()

# -------------------------
# BUILD FACTOR-LEVEL ITEM LISTS
# -------------------------

factor_item_map = {}
for factor in [f"Factor{i+1}" for i in range(N_FACTORS)]:
    factor_items = retained_items_df.loc[
        retained_items_df["Assigned_Factor"] == factor, "Item"
    ].tolist()
    factor_item_map[factor] = factor_items

# -------------------------
# CRONBACH'S ALPHA
# -------------------------

alpha_rows = []

for factor, items in factor_item_map.items():
    if len(items) >= 2:
        alpha = cronbach_alpha(item_df[items])
    else:
        alpha = np.nan

    alpha_rows.append({
        "Factor": factor,
        "n_items": len(items),
        "Alpha": alpha
    })

alpha_df = pd.DataFrame(alpha_rows)

# -------------------------
# OUTPUT TABLES
# -------------------------

print("\n" + "="*70)
print("ROTATED FACTOR LOADING MATRIX")
print("="*70)
print(loading_matrix.round(3))

print("\n" + "="*70)
print("ITEM RETENTION DECISIONS")
print("="*70)
print(retention_df)

print("\n" + "="*70)
print("FACTOR VARIANCE SUMMARY")
print("="*70)
print(variance_df.round(3))

if phi is not None:
    print("\n" + "="*70)
    print("FACTOR CORRELATION MATRIX (OBLIQUE ROTATION)")
    print("="*70)
    print(phi.round(3))

print("\n" + "="*70)
print("CRONBACH'S ALPHA BY FACTOR")
print("="*70)
print(alpha_df.round(3))

# Save outputs
loading_matrix.round(3).to_csv("efa_loadings.csv")
retention_df.to_csv("efa_item_retention.csv", index=False)
variance_df.round(3).to_csv("efa_variance_summary.csv")
alpha_df.round(3).to_csv("efa_alpha_summary.csv", index=False)

if phi is not None:
    phi.round(3).to_csv("efa_factor_correlations.csv")

# -------------------------
# APA-STYLE WRITE-UP
# -------------------------

n = item_df.shape[0]
k = item_df.shape[1]

retained_total = retained_items_df["Retain"].sum()

factor_summaries = []
for _, row in alpha_df.iterrows():
    factor = row["Factor"]
    n_items = int(row["n_items"])
    alpha = row["Alpha"]

    items = factor_item_map[factor]
    item_list = ", ".join(items) if items else "no retained items"

    factor_summaries.append(
        f"{factor} was defined by {n_items} item{'s' if n_items != 1 else ''} "
        f"({item_list}) and demonstrated internal consistency of α = {apa_alpha(alpha)}."
    )

variance_text = []
for factor, row in variance_df.iterrows():
    variance_text.append(
        f"{factor} accounted for {row['Proportion_Var']*100:.2f}% of the variance"
    )

writeup = f"""
Exploratory factor analysis was conducted on the items from the newly developed AI readiness scale using a forced six-factor solution.
Following Ford et al. (1986), the analysis used an oblique rotation to allow the factors to correlate. The extraction was conducted in Python
using a common-factor analytic approach with {ROTATION} rotation. The analysis was based on {n} participants and {k} items.

A loading of .30 was used as the minimum cutoff for interpreting an item as defining a factor. Items were not retained if they failed to load
at .30 or greater on any factor, exhibited high cross-loadings, or were theoretically inconsistent with their a priori factor assignments.
Using these criteria, {retained_total} items were retained across the six factors.

The rotated factor solution indicated that {", ".join(variance_text[:-1])}, and {variance_text[-1]}.

{" ".join(factor_summaries)}

Taken together, the results provided an initial evaluation of the scale's latent structure and internal consistency. The retained items generally
conformed to the expected six-factor framework, subject to the empirical and theoretical retention criteria described above.
"""

print("\n" + "="*70)
print("APA-STYLE WRITE-UP")
print("="*70)
print(writeup.strip())

with open("efa_apa_writeup.txt", "w", encoding="utf-8") as f:
    f.write(writeup.strip())

print("\nFiles saved:")
print("- efa_loadings.csv")
print("- efa_item_retention.csv")
print("- efa_variance_summary.csv")
print("- efa_alpha_summary.csv")
if phi is not None:
    print("- efa_factor_correlations.csv")
print("- efa_apa_writeup.txt")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



ROTATED FACTOR LOADING MATRIX
      Factor1  Factor2  Factor3  Factor4  Factor5  Factor6
ALU1    0.067   -0.030    0.797   -0.014    0.036    0.071
ALU2   -0.006    0.040    0.757   -0.038   -0.038    0.096
ALU3    0.053   -0.114    0.741    0.008    0.010    0.031
ALU4   -0.162    0.286    0.307   -0.028    0.090   -0.045
ALU5    0.498   -0.029    0.418   -0.009   -0.085    0.007
AET1    0.809   -0.007    0.012   -0.054    0.056   -0.052
AET2    0.802   -0.036   -0.008   -0.017    0.026   -0.026
AET3    0.744    0.056   -0.037   -0.100    0.052    0.016
AET4    0.394   -0.042    0.070    0.080    0.082   -0.078
AET5    0.427    0.102   -0.095    0.020   -0.095    0.409
CEJ1    0.060   -0.016    0.035   -0.003    0.044    0.772
CEJ2   -0.043   -0.035    0.053   -0.064    0.020    0.783
CEJ3   -0.042   -0.046   -0.003    0.075    0.070    0.702
CEJ4   -0.083    0.048    0.147    0.006   -0.086    0.415
CEJ5   -0.082    0.031   -0.138    0.034    0.460    0.476
RDG1   -0.012   -0.036   

Study 2: Confirmatory Factor Analysis

In [ ]:
import itertools
import math
import os
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import chi2 as chi2_dist
from semopy import Model, calc_stats


# =========================
# Configuration
# =========================
FACTOR_ITEMS = {
    "AET": ["AET1", "AET2", "AET3", "AET4"],
    "CLA": ["CLA1", "CLA2", "CLA3", "CLA4"],
    "ALU": ["ALU1", "ALU2", "ALU3", "ALU4"],
    "RDG": ["RDG1", "RDG2", "RDG3", "RDG4"],
    "WHC": ["WHC1", "WHC2", "WHC3", "WHC4"],
    "CEJ": ["CEJ1", "CEJ2", "CEJ3", "CEJ4"],
}

FACTOR_ORDER = ["AET", "CLA", "ALU", "RDG", "WHC", "CEJ"]
ALL_ITEMS = [item for factor in FACTOR_ORDER for item in FACTOR_ITEMS[factor]]

BASELINE_OUTPUT_FILES = {
    "baseline_fit": "cfa_baseline_fit.csv",
    "standardized_loadings": "cfa_standardized_loadings.csv",
    "cr_ave": "cfa_composite_reliability_ave.csv",
    "single_fit": "cfa_single_factor_fit.csv",
    "single_comparison": "cfa_single_factor_comparison.csv",
    "discriminant_models": "cfa_discriminant_validity_models.csv",
    "loading_table": "cfa_loading_table.csv",
    "fit_table": "cfa_fit_table.csv",
    "apa_writeup": "cfa_apa_writeup.txt",
}


# =========================
# Helper functions
# =========================
def build_cfa_model(factor_items: Dict[str, List[str]]) -> str:
    """
    Build a correlated CFA model syntax string in semopy/lavaan-style syntax.
    """
    lines = [f"{factor} =~ {' + '.join(items)}" for factor, items in factor_items.items()]
    return "\n".join(lines)


def build_single_factor_model(all_items: List[str], general_factor: str = "GENERAL") -> str:
    """
    Build a single-factor alternative model.
    """
    return f"{general_factor} =~ {' + '.join(all_items)}"


def build_collapsed_pairwise_model(
    factor_items: Dict[str, List[str]],
    pair: Tuple[str, str],
    merged_label: str = None
) -> str:
    """
    Build a model where two factors are collapsed into one merged factor
    and all other factors remain separate.
    """
    f1, f2 = pair
    merged_label = merged_label or f"{f1}_{f2}"
    lines = []
    merged_items = factor_items[f1] + factor_items[f2]
    lines.append(f"{merged_label} =~ {' + '.join(merged_items)}")

    for factor in FACTOR_ORDER:
        if factor not in pair:
            lines.append(f"{factor} =~ {' + '.join(factor_items[factor])}")

    return "\n".join(lines)


def fit_semopy_model(model_syntax: str, data: pd.DataFrame):
    """
    Fit a semopy model and return the model object and fit statistics DataFrame.
    """
    model = Model(model_syntax)
    model.fit(data)
    stats = calc_stats(model)
    return model, stats


def _safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def extract_fit_indices(stats_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """
    Extract key fit indices from semopy calc_stats output.
    """
    row = stats_df.iloc[0]
    out = pd.DataFrame(
        [{
            "model": model_name,
            "chi_square": _safe_float(row.get("chi2", np.nan)),
            "df": _safe_float(row.get("DoF", np.nan)),
            "p_value": _safe_float(row.get("chi2 p-value", np.nan)),
            "CFI": _safe_float(row.get("CFI", np.nan)),
            "TLI": _safe_float(row.get("TLI", np.nan)),  # TLI corresponds to NNFI
            "RMSEA": _safe_float(row.get("RMSEA", np.nan)),
            "AIC": _safe_float(row.get("AIC", np.nan)),
            "BIC": _safe_float(row.get("BIC", np.nan)),
        }]
    )
    return out


def chi_square_difference_test(
    baseline_fit: pd.DataFrame,
    nested_fit: pd.DataFrame
) -> pd.DataFrame:
    """
    Compare nested model to baseline model using chi-square difference testing.
    Assumes nested model is the more constrained model.
    """
    chi2_base = float(baseline_fit.loc[0, "chi_square"])
    df_base = float(baseline_fit.loc[0, "df"])
    chi2_nested = float(nested_fit.loc[0, "chi_square"])
    df_nested = float(nested_fit.loc[0, "df"])

    delta_chi2 = chi2_nested - chi2_base
    delta_df = df_nested - df_base

    if delta_df > 0 and delta_chi2 >= 0:
        p_diff = chi2_dist.sf(delta_chi2, delta_df)
    else:
        p_diff = np.nan

    delta_cfi = float(nested_fit.loc[0, "CFI"]) - float(baseline_fit.loc[0, "CFI"])

    return pd.DataFrame(
        [{
            "baseline_model": baseline_fit.loc[0, "model"],
            "comparison_model": nested_fit.loc[0, "model"],
            "baseline_chi_square": chi2_base,
            "baseline_df": df_base,
            "comparison_chi_square": chi2_nested,
            "comparison_df": df_nested,
            "chi_square_difference": delta_chi2,
            "df_difference": delta_df,
            "p_value_difference": p_diff,
            "delta_CFI": delta_cfi,
        }]
    )


def _find_standardized_column(df: pd.DataFrame) -> str:
    candidates = ["Est. Std", "Est.Std", "Std. Est", "Std.Est", "Estimate Std", "standardized_estimate"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        "Could not locate standardized estimate column in semopy inspect output. "
        f"Available columns: {list(df.columns)}"
    )


def _find_pvalue_column(df: pd.DataFrame) -> str:
    candidates = ["p-value", "p_value", "pvalue", "P-value", "PValue"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        "Could not locate p-value column in semopy inspect output. "
        f"Available columns: {list(df.columns)}"
    )


def _find_estimate_column(df: pd.DataFrame) -> str:
    candidates = ["Estimate", "estimate", "Est"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        "Could not locate estimate column in semopy inspect output. "
        f"Available columns: {list(df.columns)}"
    )


def extract_standardized_loadings(
    model,
    factor_items: Dict[str, List[str]]
) -> pd.DataFrame:
    """
    Extract standardized factor loadings and significance information from semopy.
    """
    est = model.inspect(std_est=True)
    std_col = _find_standardized_column(est)
    p_col = _find_pvalue_column(est)
    est_col = _find_estimate_column(est)

    records = []
    factor_set = set(factor_items.keys())
    item_set = {item for items in factor_items.values() for item in items}

    for _, row in est.iterrows():
        op = str(row.get("op", ""))
        lval = str(row.get("lval", ""))
        rval = str(row.get("rval", ""))

        # In semopy, CFA loadings typically appear as observed ~ latent
        if op == "~" and lval in item_set and rval in factor_set:
            records.append({
                "factor": rval,
                "item": lval,
                "unstandardized_loading": _safe_float(row.get(est_col, np.nan)),
                "standardized_loading": _safe_float(row.get(std_col, np.nan)),
                "SE": _safe_float(row.get("Std. Err", np.nan)),
                "z_value": _safe_float(row.get("z-value", np.nan)),
                "p_value": _safe_float(row.get(p_col, np.nan)),
            })

    loadings = pd.DataFrame(records)
    if loadings.empty:
        raise ValueError("No factor loadings were extracted. Check model syntax and semopy output.")

    loadings["significant"] = loadings["p_value"] < 0.05

    factor_order_map = {f: i for i, f in enumerate(FACTOR_ORDER)}
    item_order_map = {item: i for i, item in enumerate(ALL_ITEMS)}
    loadings["factor_order"] = loadings["factor"].map(factor_order_map)
    loadings["item_order"] = loadings["item"].map(item_order_map)
    loadings = loadings.sort_values(["factor_order", "item_order"]).drop(columns=["factor_order", "item_order"])
    loadings.reset_index(drop=True, inplace=True)
    return loadings


def summarize_loading_significance(loadings_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize loading significance and standardized loading distribution.
    """
    all_sig = bool(loadings_df["significant"].all())
    out = pd.DataFrame(
        [{
            "n_loadings": int(loadings_df.shape[0]),
            "all_loadings_significant": all_sig,
            "n_significant": int(loadings_df["significant"].sum()),
            "mean_standardized_loading": loadings_df["standardized_loading"].mean(),
            "minimum_standardized_loading": loadings_df["standardized_loading"].min(),
            "maximum_standardized_loading": loadings_df["standardized_loading"].max(),
        }]
    )
    return out


def compute_cr_ave(loadings_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute composite reliability (CR) and average variance extracted (AVE)
    from standardized loadings.
    """
    rows = []
    for factor in FACTOR_ORDER:
        sub = loadings_df.loc[loadings_df["factor"] == factor].copy()
        lambdas = sub["standardized_loading"].astype(float).values
        thetas = 1 - (lambdas ** 2)

        sum_lambda = np.sum(lambdas)
        sum_lambda_sq = np.sum(lambdas ** 2)
        sum_theta = np.sum(thetas)

        cr = (sum_lambda ** 2) / ((sum_lambda ** 2) + sum_theta) if ((sum_lambda ** 2) + sum_theta) > 0 else np.nan
        ave = sum_lambda_sq / (sum_lambda_sq + sum_theta) if (sum_lambda_sq + sum_theta) > 0 else np.nan

        rows.append({
            "factor": factor,
            "n_items": len(lambdas),
            "mean_standardized_loading": np.mean(lambdas),
            "minimum_standardized_loading": np.min(lambdas),
            "maximum_standardized_loading": np.max(lambdas),
            "CR": cr,
            "AVE": ave,
        })

    return pd.DataFrame(rows)


def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.3f}".replace("0.", ".")


def write_apa_summary(
    filepath: str,
    n: int,
    baseline_fit: pd.DataFrame,
    loading_summary: pd.DataFrame,
    cr_ave_df: pd.DataFrame,
    single_factor_fit: pd.DataFrame,
    single_factor_comp: pd.DataFrame,
    discriminant_df: pd.DataFrame
):
    """
    Write an APA-style summary of the CFA results.
    """
    b = baseline_fit.iloc[0]
    ls = loading_summary.iloc[0]
    sf = single_factor_fit.iloc[0]
    sfc = single_factor_comp.iloc[0]

    cr_text = "; ".join(
        [
            f"{row['factor']}: CR = {row['CR']:.3f}, AVE = {row['AVE']:.3f}"
            for _, row in cr_ave_df.iterrows()
        ]
    )

    dv_sig = discriminant_df["p_value_difference"].lt(0.05).sum()
    dv_total = discriminant_df.shape[0]
    dv_best = discriminant_df.sort_values("p_value_difference", ascending=True).iloc[0]
    dv_worst_cfi = discriminant_df["delta_CFI"].max()

    text = f"""Confirmatory factor analysis (CFA) was conducted on the 24 retained items of the AI readiness scale using semopy with maximum likelihood estimation and listwise deletion of missing data (N = {n}). The hypothesized measurement model specified six correlated latent factors labeled AET, CLA, ALU, RDG, WHC, and CEJ, with four indicators loading on each factor.

The six-factor model showed the following fit: χ²({int(b['df'])}) = {b['chi_square']:.3f}, p {format_p(b['p_value'])}, CFI = {b['CFI']:.3f}, TLI/NNFI = {b['TLI']:.3f}, RMSEA = {b['RMSEA']:.3f}, AIC = {b['AIC']:.3f}, and BIC = {b['BIC']:.3f}. Standardized factor loadings were all statistically significant: {bool(ls['all_loadings_significant'])}. The average standardized loading was {ls['mean_standardized_loading']:.3f}, with a minimum of {ls['minimum_standardized_loading']:.3f} and a maximum of {ls['maximum_standardized_loading']:.3f}.

Composite reliability and average variance extracted supported the internal consistency and convergent validity of the factors. The factor-level estimates were as follows: {cr_text}.

A more restrictive single-factor alternative model was also estimated to evaluate whether the 24 items were adequately represented by a single general construct. The single-factor model fit was χ²({int(sf['df'])}) = {sf['chi_square']:.3f}, p {format_p(sf['p_value'])}, CFI = {sf['CFI']:.3f}, TLI/NNFI = {sf['TLI']:.3f}, and RMSEA = {sf['RMSEA']:.3f}. Relative to the six-factor model, the single-factor model showed a chi-square increase of Δχ²({int(sfc['df_difference'])}) = {sfc['chi_square_difference']:.3f}, p {format_p(sfc['p_value_difference'])}, with ΔCFI = {sfc['delta_CFI']:.3f}. This pattern indicates that the six-factor representation fit the data better than the unidimensional alternative.

Discriminant validity was further evaluated using 15 pairwise nested comparisons in which each possible pair of latent factors was collapsed into a single merged factor while the remaining factors were left unchanged. Across these 15 comparisons, {dv_sig} of {dv_total} collapsed models fit significantly worse than the baseline six-factor model according to the chi-square difference test. The strongest evidence against collapsing factors was observed for the {dv_best['factor_pair_tested']} comparison, which yielded Δχ²({int(dv_best['df_difference'])}) = {dv_best['chi_square_difference']:.3f}, p {format_p(dv_best['p_value_difference'])}, with ΔCFI = {dv_best['delta_CFI']:.3f}. Across all pairwise tests, the largest ΔCFI (collapsed minus baseline) was {dv_worst_cfi:.3f}. Taken together, these nested comparisons provide evidence that the six latent dimensions are empirically distinguishable.

Overall, the results support the psychometric adequacy of the six-factor AI readiness measure. The hypothesized six-factor correlated structure demonstrated acceptable model fit, all indicators loaded significantly on their intended factors, reliability and convergent validity estimates were adequate at the construct level, the six-factor model outperformed a single-factor alternative, and the pairwise discriminant validity tests were generally consistent with the distinctiveness of the proposed dimensions.
"""
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)


def print_section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


# =========================
# Main analysis
# =========================
def main():
    # 1. Load CSV
    print_section("Loading data")
    data = pd.read_csv("CFA_ai_readiness_data.csv")
    print(f"Original data shape: {data.shape}")

    # 2. Keep only the 24 modeled items
    missing_cols = [c for c in ALL_ITEMS if c not in data.columns]
    if missing_cols:
        raise ValueError(f"The following required columns are missing from ai_readiness_data.csv: {missing_cols}")

    data = data[ALL_ITEMS].copy()
    print(f"Retained modeled items: {len(ALL_ITEMS)}")
    print(f"Data shape after retaining modeled items: {data.shape}")

    # 3. Listwise deletion
    data = data.dropna(axis=0).reset_index(drop=True)
    print(f"Data shape after listwise deletion: {data.shape}")

    # 4. Fit six-factor correlated CFA model
    print_section("Fitting six-factor baseline CFA model")
    baseline_model_syntax = build_cfa_model(FACTOR_ITEMS)
    baseline_model, baseline_stats = fit_semopy_model(baseline_model_syntax, data)
    baseline_fit = extract_fit_indices(baseline_stats, "Six-factor baseline")
    print(baseline_fit.to_string(index=False))

    # 5. Fit indices already extracted above

    # 6. Standardized loadings and significance
    print_section("Extracting standardized loadings")
    standardized_loadings = extract_standardized_loadings(baseline_model, FACTOR_ITEMS)
    loading_summary = summarize_loading_significance(standardized_loadings)
    print(standardized_loadings.to_string(index=False))
    print("\nLoading summary:")
    print(loading_summary.to_string(index=False))

    # 7. Mean/min/max standardized loading already computed above

    # 8. Composite reliability and AVE
    print_section("Computing composite reliability (CR) and AVE")
    cr_ave_df = compute_cr_ave(standardized_loadings)
    print(cr_ave_df.to_string(index=False))

    # 9. Single-factor model
    print_section("Fitting single-factor alternative model")
    single_factor_syntax = build_single_factor_model(ALL_ITEMS, general_factor="GENERAL")
    single_model, single_stats = fit_semopy_model(single_factor_syntax, data)
    single_factor_fit = extract_fit_indices(single_stats, "Single-factor alternative")
    print(single_factor_fit.to_string(index=False))

    # 10. Compare baseline vs single-factor
    print_section("Comparing baseline model to single-factor model")
    single_factor_comparison = chi_square_difference_test(baseline_fit, single_factor_fit)
    print(single_factor_comparison.to_string(index=False))

    # 11. Discriminant validity tests: 15 pairwise collapsed models
    print_section("Fitting discriminant validity pairwise collapsed models")
    pairwise_results = []

    for pair in itertools.combinations(FACTOR_ORDER, 2):
        pair_label = f"{pair[0]} vs {pair[1]}"
        merged_label = f"{pair[0]}_{pair[1]}"
        syntax = build_collapsed_pairwise_model(FACTOR_ITEMS, pair, merged_label=merged_label)
        model_name = f"Collapsed {pair[0]}_{pair[1]}"

        try:
            model, stats = fit_semopy_model(syntax, data)
            fit_df = extract_fit_indices(stats, model_name)
            comp_df = chi_square_difference_test(baseline_fit, fit_df)

            pairwise_results.append({
                "factor_pair_tested": pair_label,
                "model": model_name,
                "chi_square": float(fit_df.loc[0, "chi_square"]),
                "df": float(fit_df.loc[0, "df"]),
                "p_value": float(fit_df.loc[0, "p_value"]),
                "CFI": float(fit_df.loc[0, "CFI"]),
                "TLI": float(fit_df.loc[0, "TLI"]),
                "RMSEA": float(fit_df.loc[0, "RMSEA"]),
                "chi_square_difference": float(comp_df.loc[0, "chi_square_difference"]),
                "df_difference": float(comp_df.loc[0, "df_difference"]),
                "p_value_difference": float(comp_df.loc[0, "p_value_difference"]),
                "delta_CFI": float(comp_df.loc[0, "delta_CFI"]),
                "AIC": float(fit_df.loc[0, "AIC"]),
                "BIC": float(fit_df.loc[0, "BIC"]),
            })

            print(f"\n{pair_label}")
            print(fit_df.to_string(index=False))
            print(comp_df.to_string(index=False))

        except Exception as e:
            pairwise_results.append({
                "factor_pair_tested": pair_label,
                "model": model_name,
                "chi_square": np.nan,
                "df": np.nan,
                "p_value": np.nan,
                "CFI": np.nan,
                "TLI": np.nan,
                "RMSEA": np.nan,
                "chi_square_difference": np.nan,
                "df_difference": np.nan,
                "p_value_difference": np.nan,
                "delta_CFI": np.nan,
                "AIC": np.nan,
                "BIC": np.nan,
                "error": str(e),
            })
            print(f"\n{pair_label}")
            print(f"Model failed: {e}")

    discriminant_df = pd.DataFrame(pairwise_results)

    # 12. Output pairwise results done above

    # 13. Save results to CSV files
    print_section("Saving output files")
    baseline_fit.to_csv(BASELINE_OUTPUT_FILES["baseline_fit"], index=False)
    standardized_loadings.to_csv(BASELINE_OUTPUT_FILES["standardized_loadings"], index=False)
    cr_ave_df.to_csv(BASELINE_OUTPUT_FILES["cr_ave"], index=False)
    single_factor_fit.to_csv(BASELINE_OUTPUT_FILES["single_fit"], index=False)
    single_factor_comparison.to_csv(BASELINE_OUTPUT_FILES["single_comparison"], index=False)
    discriminant_df.to_csv(BASELINE_OUTPUT_FILES["discriminant_models"], index=False)

    # Requested summary tables
    loading_table = standardized_loadings.copy()
    fit_table = pd.concat(
        [
            baseline_fit,
            single_factor_fit,
            discriminant_df[["model", "chi_square", "df", "p_value", "CFI", "TLI", "RMSEA", "AIC", "BIC"]].copy(),
        ],
        axis=0,
        ignore_index=True
    )
    loading_table.to_csv(BASELINE_OUTPUT_FILES["loading_table"], index=False)
    fit_table.to_csv(BASELINE_OUTPUT_FILES["fit_table"], index=False)

    for key, filepath in BASELINE_OUTPUT_FILES.items():
        if key != "apa_writeup":
            print(f"Saved: {filepath}")

    # 14. Generate APA-style write-up
    print_section("Writing APA-style summary")
    write_apa_summary(
        filepath=BASELINE_OUTPUT_FILES["apa_writeup"],
        n=data.shape[0],
        baseline_fit=baseline_fit,
        loading_summary=loading_summary,
        cr_ave_df=cr_ave_df,
        single_factor_fit=single_factor_fit,
        single_factor_comp=single_factor_comparison,
        discriminant_df=discriminant_df.dropna(subset=["chi_square_difference", "df_difference"])
    )
    print(f"Saved: {BASELINE_OUTPUT_FILES['apa_writeup']}")

    # 15. Final console summary
    print_section("Final console summary")
    print("Baseline fit:")
    print(baseline_fit.to_string(index=False))

    print("\nLoading summary:")
    print(loading_summary.to_string(index=False))

    print("\nCR and AVE:")
    print(cr_ave_df.to_string(index=False))

    print("\nSingle-factor comparison:")
    print(single_factor_comparison.to_string(index=False))

    print("\nTop discriminant validity results (first 15 rows):")
    print(discriminant_df.to_string(index=False))


if __name__ == "__main__":
    main()


Loading data
Original data shape: (800, 24)
Retained modeled items: 24
Data shape after retaining modeled items: (800, 24)
Data shape after listwise deletion: (800, 24)

Fitting six-factor baseline CFA model
              model  chi_square    df  p_value      CFI      TLI    RMSEA        AIC        BIC
Six-factor baseline  315.942038 237.0 0.000455 0.980482 0.977271 0.020418 125.210145 420.340684

Extracting standardized loadings
factor item  unstandardized_loading  standardized_loading       SE   z_value      p_value  significant
   AET AET1                1.000000              0.755629      NaN       NaN          NaN        False
   AET AET2                0.922820              0.726560 0.057539 16.038122 0.000000e+00         True
   AET AET3                0.864584              0.662736 0.056336 15.346851 0.000000e+00         True
   AET AET4                0.429649              0.327038 0.053201  8.075958 6.661338e-16         True
   CLA CLA1                1.000000              0

Phase 3 - Convergent, Discriminant, and Incremental Validity

In [ ]:
"""
Phase 3 validation analyses for a six-factor, 24-item AI readiness scale.
"""

import os
import math
import warnings
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from semopy import Model, calc_stats

warnings.filterwarnings("ignore")

DATA_FILE = "ai_readiness_validity_data.csv"

AI_FACTORS = {
    "AET": ["AET1", "AET2", "AET3", "AET4"],
    "CLA": ["CLA1", "CLA2", "CLA3", "CLA4"],
    "ALU": ["ALU1", "ALU2", "ALU3", "ALU4"],
    "RDG": ["RDG1", "RDG2", "RDG3", "RDG4"],
    "WHC": ["WHC1", "WHC2", "WHC3", "WHC4"],
    "CEJ": ["CEJ1", "CEJ2", "CEJ3", "CEJ4"],
}

CONVERGENT_VARS = [
    "ai_self_efficacy",
    "prompt_engineering_confidence",
]

MATCHED_CONVERGENT = {
    "AET": ["ai_self_efficacy"],
    "CLA": ["ai_self_efficacy", "prompt_engineering_confidence"],
    "ALU": [],
    "RDG": ["prompt_engineering_confidence"],
    "WHC": [],
    "CEJ": ["prompt_engineering_confidence"],
}

DISCRIMINANT_VARS = [
    "political_interest",
    "social_desirability",
]

SOCIAL_DESIRABILITY_VARS = [
    "social_desirability",
]

ALTERNATIVE_LATENT_FACTORS = {
    "AISE": ["AISE1", "AISE2", "AISE3", "AISE4"],
    "PEC": ["PEC1", "PEC2", "PEC3", "PEC4"],
    "PI": ["PI1", "PI2", "PI3", "PI4"],
    "SD": ["SD1", "SD2", "SD3", "SD4"],
}

BASELINE_PREDICTORS = [
    "general_cognitive_ability",
    "prior_coding_experience_years",
    "programming_self_efficacy",
    "coding_knowledge_test",
]

CRITERION_VARS = [
    "coding_accuracy",
    "debugging_score",
    "code_quality_rating",
    "coding_efficiency",
    "coding_performance_task",
]

OUTPUT_FILES = {
    "scored_data": "ai_readiness_scored_data.csv",
    "conv_corr": "convergent_validity_correlations.csv",
    "conv_match": "convergent_validity_matched_vs_nonmatched.csv",
    "disc_corr": "discriminant_validity_correlations.csv",
    "disc_latent": "discriminant_validity_latent_model_comparisons.csv",
    "inc_steps": "incremental_validity_model_steps.csv",
    "inc_comp": "incremental_validity_model_comparisons.csv",
    "inc_coef": "incremental_validity_full_model_coefficients.csv",
    "conv_summary": "phase3_convergent_summary.csv",
    "disc_summary": "phase3_discriminant_summary.csv",
    "inc_summary": "phase3_incremental_summary.csv",
    "all_overview": "phase3_all_results_overview.csv",
    "apa_writeup": "phase3_apa_writeup.txt",
}


def safe_float(x: Any) -> float:
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def safe_str_num(x: Any, digits: int = 3) -> str:
    val = safe_float(x)
    if np.isnan(val):
        return "NaN"
    return f"{val:.{digits}f}"


def ensure_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def check_missing_variables(df: pd.DataFrame, variables: List[str], context: str = "") -> List[str]:
    missing = [v for v in variables if v not in df.columns]
    if missing:
        ctx = f" for {context}" if context else ""
        print(f"WARNING: Missing variables{ctx}: {missing}")
    return missing


def listwise_delete(df: pd.DataFrame, variables: List[str]) -> pd.DataFrame:
    available = [v for v in variables if v in df.columns]
    if not available:
        return pd.DataFrame()
    return df[available].dropna(axis=0, how="any").copy()


def label_effect_size_r(r: float) -> str:
    if pd.isna(r):
        return "not estimable"
    ar = abs(r)
    if ar < 0.30:
        return "small"
    if ar < 0.50:
        return "medium"
    return "large"


def fisher_z(r: float) -> float:
    if pd.isna(r):
        return np.nan
    r_clipped = np.clip(r, -0.999999, 0.999999)
    return 0.5 * np.log((1 + r_clipped) / (1 - r_clipped))


def compute_zero_order_correlations(
    df: pd.DataFrame,
    predictors: List[str],
    outcomes: List[str],
    analysis_label: str
) -> pd.DataFrame:
    rows = []
    for x in predictors:
        for y in outcomes:
            if x not in df.columns or y not in df.columns:
                print(f"WARNING: Skipping correlation {x} with {y}; variable missing.")
                continue
            sub = listwise_delete(df, [x, y])
            n = len(sub)
            if n < 3:
                print(f"WARNING: Skipping correlation {x} with {y}; insufficient n ({n}).")
                rows.append({
                    "analysis": analysis_label,
                    "predictor": x,
                    "outcome": y,
                    "n": n,
                    "r": np.nan,
                    "p_value": np.nan,
                    "effect_size_label": "not estimable",
                })
                continue
            try:
                r, p = stats.pearsonr(sub[x], sub[y])
            except Exception as e:
                print(f"WARNING: Correlation failed for {x} with {y}: {e}")
                r, p = np.nan, np.nan
            rows.append({
                "analysis": analysis_label,
                "predictor": x,
                "outcome": y,
                "n": n,
                "r": r,
                "p_value": p,
                "effect_size_label": label_effect_size_r(r),
            })
    return pd.DataFrame(rows)


def compare_matched_vs_nonmatched_convergent(
    df: pd.DataFrame,
    factors: List[str],
    convergent_vars: List[str],
    matched_dict: Dict[str, List[str]]
) -> pd.DataFrame:
    rows = []
    for factor in factors:
        matched_vars = matched_dict.get(factor, [])
        nonmatched_vars = [v for v in convergent_vars if v not in matched_vars]

        required_vars = [factor] + list(set(matched_vars + nonmatched_vars))
        missing = check_missing_variables(df, required_vars, context=f"matched vs nonmatched for {factor}")
        usable_required = [v for v in required_vars if v not in missing]

        if factor not in usable_required:
            rows.append({
                "factor": factor,
                "n": 0,
                "matched_variables": ", ".join(matched_vars),
                "nonmatched_variables": ", ".join(nonmatched_vars),
                "k_matched": 0,
                "k_nonmatched": 0,
                "avg_r_matched": np.nan,
                "avg_r_nonmatched": np.nan,
                "avg_z_matched": np.nan,
                "avg_z_nonmatched": np.nan,
                "z_approx": np.nan,
                "p_value_approx": np.nan,
                "interpretation": "not estimable",
                "note": "Fisher r-to-z comparison is an approximation.",
            })
            continue

        sub = listwise_delete(df, usable_required)
        n = len(sub)

        matched_rs = []
        for v in matched_vars:
            if v in sub.columns and n >= 3:
                try:
                    r, _ = stats.pearsonr(sub[factor], sub[v])
                    matched_rs.append(r)
                except Exception:
                    pass

        nonmatched_rs = []
        for v in nonmatched_vars:
            if v in sub.columns and n >= 3:
                try:
                    r, _ = stats.pearsonr(sub[factor], sub[v])
                    nonmatched_rs.append(r)
                except Exception:
                    pass

        avg_r_matched = np.mean(matched_rs) if len(matched_rs) > 0 else np.nan
        avg_r_nonmatched = np.mean(nonmatched_rs) if len(nonmatched_rs) > 0 else np.nan
        avg_z_matched = np.mean([fisher_z(r) for r in matched_rs]) if len(matched_rs) > 0 else np.nan
        avg_z_nonmatched = np.mean([fisher_z(r) for r in nonmatched_rs]) if len(nonmatched_rs) > 0 else np.nan

        z_approx = np.nan
        p_approx = np.nan
        interpretation = "not estimable"

        if n > 3 and len(matched_rs) > 0 and len(nonmatched_rs) > 0:
            try:
                se = math.sqrt((1 / (n - 3)) / len(matched_rs) + (1 / (n - 3)) / len(nonmatched_rs))
                if se > 0:
                    z_approx = (avg_z_matched - avg_z_nonmatched) / se
                    p_approx = 2 * (1 - stats.norm.cdf(abs(z_approx)))
                    if p_approx < 0.05 and avg_r_matched > avg_r_nonmatched:
                        interpretation = "matched > nonmatched (approx. significant)"
                    elif p_approx < 0.05 and avg_r_matched < avg_r_nonmatched:
                        interpretation = "matched < nonmatched (approx. significant)"
                    else:
                        interpretation = "difference not approx. significant"
            except Exception as e:
                print(f"WARNING: Fisher approximation failed for {factor}: {e}")

        rows.append({
            "factor": factor,
            "n": n,
            "matched_variables": ", ".join(matched_vars),
            "nonmatched_variables": ", ".join(nonmatched_vars),
            "k_matched": len(matched_rs),
            "k_nonmatched": len(nonmatched_rs),
            "avg_r_matched": avg_r_matched,
            "avg_r_nonmatched": avg_r_nonmatched,
            "avg_z_matched": avg_z_matched,
            "avg_z_nonmatched": avg_z_nonmatched,
            "z_approx": z_approx,
            "p_value_approx": p_approx,
            "interpretation": interpretation,
            "note": "Fisher r-to-z comparison is an approximation based on averaged transformed correlations.",
        })
    return pd.DataFrame(rows)


def build_semopy_model_string(
    ai_factors: Dict[str, List[str]],
    alternative_factor_name: Optional[str] = None,
    alternative_items: Optional[List[str]] = None,
    collapsed_factor: Optional[str] = None
) -> str:
    lines = []
    for factor, items in ai_factors.items():
        if collapsed_factor is not None and alternative_factor_name is not None and factor == collapsed_factor:
            merged_name = f"{factor}_{alternative_factor_name}_MERGED"
            merged_items = items + (alternative_items or [])
            lines.append(f"{merged_name} =~ " + " + ".join(merged_items))
        else:
            lines.append(f"{factor} =~ " + " + ".join(items))
    if alternative_factor_name is not None and alternative_items is not None and collapsed_factor is None:
        lines.append(f"{alternative_factor_name} =~ " + " + ".join(alternative_items))
    return "\n".join(lines)


def extract_semopy_fit_stats(stats_obj: Any) -> Dict[str, float]:
    result = {
        "chi2": np.nan,
        "DoF": np.nan,
        "p_value": np.nan,
        "CFI": np.nan,
        "TLI": np.nan,
        "RMSEA": np.nan,
    }
    if stats_obj is None:
        return result
    try:
        if isinstance(stats_obj, pd.DataFrame):
            for key in result.keys():
                if key in stats_obj.index and "Value" in stats_obj.columns:
                    result[key] = safe_float(stats_obj.loc[key, "Value"])
                elif key in stats_obj.columns and len(stats_obj) > 0:
                    result[key] = safe_float(stats_obj.iloc[0][key])
        elif isinstance(stats_obj, dict):
            for key in result.keys():
                if key in stats_obj:
                    result[key] = safe_float(stats_obj[key])
    except Exception as e:
        print(f"WARNING: Could not extract semopy fit statistics: {e}")
    return result


def fit_semopy_model(model_desc: str, data: pd.DataFrame) -> Tuple[Optional[Model], Dict[str, float], Optional[str]]:
    try:
        model = Model(model_desc)
        model.fit(data)
        stats_obj = calc_stats(model)
        fit_stats = extract_semopy_fit_stats(stats_obj)
        return model, fit_stats, None
    except Exception as e:
        return None, {
            "chi2": np.nan,
            "DoF": np.nan,
            "p_value": np.nan,
            "CFI": np.nan,
            "TLI": np.nan,
            "RMSEA": np.nan,
        }, str(e)


def chi_square_difference_test(
    baseline_stats: Dict[str, float],
    collapsed_stats: Dict[str, float]
) -> Tuple[float, float, float]:
    chi2_base = safe_float(baseline_stats.get("chi2"))
    df_base = safe_float(baseline_stats.get("DoF"))
    chi2_collapsed = safe_float(collapsed_stats.get("chi2"))
    df_collapsed = safe_float(collapsed_stats.get("DoF"))

    if np.isnan(chi2_base) or np.isnan(df_base) or np.isnan(chi2_collapsed) or np.isnan(df_collapsed):
        return np.nan, np.nan, np.nan

    chi2_diff = chi2_collapsed - chi2_base
    df_diff = df_collapsed - df_base

    if df_diff <= 0 or np.isnan(chi2_diff):
        return chi2_diff, df_diff, np.nan

    try:
        p_diff = 1 - stats.chi2.cdf(chi2_diff, df_diff)
    except Exception:
        p_diff = np.nan
    return chi2_diff, df_diff, p_diff


def run_discriminant_latent_tests(
    df: pd.DataFrame,
    ai_factors: Dict[str, List[str]],
    alternative_factors: Dict[str, List[str]]
) -> pd.DataFrame:
    rows = []
    all_ai_items = [item for items in ai_factors.values() for item in items]

    for alt_name, alt_items in alternative_factors.items():
        baseline_vars = list(set(all_ai_items + alt_items))
        missing = check_missing_variables(df, baseline_vars, context=f"latent discriminant baseline for {alt_name}")
        usable_vars = [v for v in baseline_vars if v not in missing]
        sub = listwise_delete(df, usable_vars)

        if len(sub) < 30:
            print(f"WARNING: Low n ({len(sub)}) for latent discriminant model with {alt_name}.")
        if len(usable_vars) < 8:
            print(f"WARNING: Too few variables available for latent discriminant model with {alt_name}.")
            for factor in ai_factors.keys():
                rows.append({
                    "alternative_factor": alt_name,
                    "ai_factor_tested": factor,
                    "n": len(sub),
                    "baseline_model_fit": "failed",
                    "collapsed_model_fit": "not attempted",
                    "baseline_chi2": np.nan,
                    "baseline_df": np.nan,
                    "baseline_p_value": np.nan,
                    "baseline_CFI": np.nan,
                    "baseline_TLI": np.nan,
                    "baseline_RMSEA": np.nan,
                    "collapsed_chi2": np.nan,
                    "collapsed_df": np.nan,
                    "collapsed_p_value": np.nan,
                    "collapsed_CFI": np.nan,
                    "collapsed_TLI": np.nan,
                    "collapsed_RMSEA": np.nan,
                    "chi2_diff": np.nan,
                    "df_diff": np.nan,
                    "p_value_diff": np.nan,
                    "delta_CFI": np.nan,
                    "interpretation": "not estimable",
                    "notes": "Insufficient observed indicators after missing variable checks.",
                })
            continue

        baseline_model_desc = build_semopy_model_string(
            ai_factors=ai_factors,
            alternative_factor_name=alt_name,
            alternative_items=alt_items,
            collapsed_factor=None
        )
        baseline_model, baseline_fit, baseline_err = fit_semopy_model(baseline_model_desc, sub)

        if baseline_err is not None:
            print(f"WARNING: Baseline semopy model failed for {alt_name}: {baseline_err}")

        for factor in ai_factors.keys():
            collapsed_model_desc = build_semopy_model_string(
                ai_factors=ai_factors,
                alternative_factor_name=alt_name,
                alternative_items=alt_items,
                collapsed_factor=factor
            )
            collapsed_model, collapsed_fit, collapsed_err = fit_semopy_model(collapsed_model_desc, sub)

            if collapsed_err is not None:
                print(f"WARNING: Collapsed semopy model failed for {factor} vs {alt_name}: {collapsed_err}")

            chi2_diff, df_diff, p_diff = chi_square_difference_test(baseline_fit, collapsed_fit)
            delta_cfi = safe_float(collapsed_fit.get("CFI")) - safe_float(baseline_fit.get("CFI"))
            interpretation = "not estimable"

            if not np.isnan(p_diff):
                if p_diff < 0.05:
                    interpretation = "collapsed model worse than baseline; supports discriminant validity"
                else:
                    interpretation = "collapsed model not significantly worse; weaker evidence for discriminant validity"

            rows.append({
                "alternative_factor": alt_name,
                "ai_factor_tested": factor,
                "n": len(sub),
                "baseline_model_fit": "ok" if baseline_err is None else "failed",
                "collapsed_model_fit": "ok" if collapsed_err is None else "failed",
                "baseline_chi2": baseline_fit.get("chi2"),
                "baseline_df": baseline_fit.get("DoF"),
                "baseline_p_value": baseline_fit.get("p_value"),
                "baseline_CFI": baseline_fit.get("CFI"),
                "baseline_TLI": baseline_fit.get("TLI"),
                "baseline_RMSEA": baseline_fit.get("RMSEA"),
                "collapsed_chi2": collapsed_fit.get("chi2"),
                "collapsed_df": collapsed_fit.get("DoF"),
                "collapsed_p_value": collapsed_fit.get("p_value"),
                "collapsed_CFI": collapsed_fit.get("CFI"),
                "collapsed_TLI": collapsed_fit.get("TLI"),
                "collapsed_RMSEA": collapsed_fit.get("RMSEA"),
                "chi2_diff": chi2_diff,
                "df_diff": df_diff,
                "p_value_diff": p_diff,
                "delta_CFI": delta_cfi,
                "interpretation": interpretation,
                "notes": (
                    "Nested latent discriminant validity comparison using semopy."
                    if baseline_err is None and collapsed_err is None
                    else f"baseline_error={baseline_err}; collapsed_error={collapsed_err}"
                ),
            })

    return pd.DataFrame(rows)


def fit_ols_model(
    df: pd.DataFrame,
    y_var: str,
    x_vars: List[str]
) -> Tuple[Optional[Any], pd.DataFrame]:
    missing = check_missing_variables(df, [y_var] + x_vars, context=f"OLS model for {y_var}")
    usable_vars = [v for v in [y_var] + x_vars if v not in missing]
    if y_var not in usable_vars:
        return None, pd.DataFrame()

    sub = listwise_delete(df, usable_vars)
    if len(sub) < max(10, len(x_vars) + 2):
        print(f"WARNING: OLS model for {y_var} with predictors {x_vars} has low n ({len(sub)}).")

    if len(sub) < 3 or len(x_vars) == 0:
        return None, sub

    try:
        X = sm.add_constant(sub[x_vars], has_constant="add")
        y = sub[y_var]
        model = sm.OLS(y, X).fit()
        return model, sub
    except Exception as e:
        print(f"WARNING: OLS model failed for {y_var} with predictors {x_vars}: {e}")
        return None, sub


def extract_model_step_row(
    model: Optional[Any],
    sub: pd.DataFrame,
    criterion: str,
    ordering: str,
    step: str,
    predictors_entered: List[str]
) -> Dict[str, Any]:
    if model is None:
        return {
            "criterion_variable": criterion,
            "model_ordering": ordering,
            "step": step,
            "predictors_entered": ", ".join(predictors_entered),
            "n": len(sub),
            "R": np.nan,
            "R_squared": np.nan,
            "adjusted_R_squared": np.nan,
            "F_statistic": np.nan,
            "model_p_value": np.nan,
        }

    r2 = safe_float(model.rsquared)
    r = np.sqrt(r2) if not np.isnan(r2) and r2 >= 0 else np.nan
    return {
        "criterion_variable": criterion,
        "model_ordering": ordering,
        "step": step,
        "predictors_entered": ", ".join(predictors_entered),
        "n": int(safe_float(model.nobs)),
        "R": r,
        "R_squared": r2,
        "adjusted_R_squared": safe_float(model.rsquared_adj),
        "F_statistic": safe_float(model.fvalue),
        "model_p_value": safe_float(model.f_pvalue),
    }


def compute_f_change(
    model_restricted: Optional[Any],
    model_full: Optional[Any]
) -> Tuple[float, float, float, float, float]:
    if model_restricted is None or model_full is None:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        r2_restricted = safe_float(model_restricted.rsquared)
        r2_full = safe_float(model_full.rsquared)
        df_full = safe_float(model_full.df_resid)
        df_num = safe_float(model_restricted.df_resid - model_full.df_resid)
        if np.isnan(r2_restricted) or np.isnan(r2_full) or np.isnan(df_full) or np.isnan(df_num):
            return np.nan, np.nan, np.nan, np.nan, np.nan
        delta_r2 = r2_full - r2_restricted
        if df_num <= 0 or df_full <= 0:
            return delta_r2, np.nan, df_num, df_full, np.nan
        numerator = delta_r2 / df_num
        denominator = (1 - r2_full) / df_full
        if denominator <= 0:
            return delta_r2, np.nan, df_num, df_full, np.nan
        f_change = numerator / denominator
        p_change = 1 - stats.f.cdf(f_change, df_num, df_full)
        return delta_r2, f_change, df_num, df_full, p_change
    except Exception as e:
        print(f"WARNING: F-change computation failed: {e}")
        return np.nan, np.nan, np.nan, np.nan, np.nan


def fit_standardized_betas(
    df: pd.DataFrame,
    y_var: str,
    x_vars: List[str]
) -> pd.DataFrame:
    missing = check_missing_variables(df, [y_var] + x_vars, context=f"standardized betas for {y_var}")
    usable_vars = [v for v in [y_var] + x_vars if v not in missing]
    if y_var not in usable_vars or len([v for v in x_vars if v in usable_vars]) == 0:
        return pd.DataFrame()

    x_usable = [v for v in x_vars if v in usable_vars]
    sub = listwise_delete(df, [y_var] + x_usable)
    if len(sub) < 3:
        return pd.DataFrame()

    try:
        zdf = sub[[y_var] + x_usable].copy()
        for col in zdf.columns:
            sd = zdf[col].std(ddof=1)
            if pd.isna(sd) or sd == 0:
                zdf[col] = np.nan
            else:
                zdf[col] = (zdf[col] - zdf[col].mean()) / sd
        zdf = zdf.dropna()
        if len(zdf) < 3:
            return pd.DataFrame()

        X = sm.add_constant(zdf[x_usable], has_constant="add")
        y = zdf[y_var]
        model = sm.OLS(y, X).fit()

        rows = []
        for param in model.params.index:
            if param == "const":
                continue
            rows.append({
                "predictor": param,
                "standardized_beta": safe_float(model.params.get(param)),
                "standard_error": safe_float(model.bse.get(param)),
                "t_value": safe_float(model.tvalues.get(param)),
                "p_value": safe_float(model.pvalues.get(param)),
                "n": int(safe_float(model.nobs)),
            })
        return pd.DataFrame(rows)
    except Exception as e:
        print(f"WARNING: Standardized beta model failed for {y_var}: {e}")
        return pd.DataFrame()


def run_incremental_validity(
    df: pd.DataFrame,
    readiness_factors: List[str],
    baseline_predictors: List[str],
    criterion_vars: List[str]
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    step_rows = []
    comp_rows = []
    coef_rows = []

    for criterion in criterion_vars:
        print(f"\nRunning incremental validity for criterion: {criterion}")

        ord1_step1_vars = baseline_predictors.copy()
        ord1_step2_vars = baseline_predictors + readiness_factors

        model1_step1, sub1 = fit_ols_model(df, criterion, ord1_step1_vars)
        model1_step2, sub2 = fit_ols_model(df, criterion, ord1_step2_vars)

        step_rows.append(extract_model_step_row(model1_step1, sub1, criterion, "Ordering 1", "Step 1", ord1_step1_vars))
        step_rows.append(extract_model_step_row(model1_step2, sub2, criterion, "Ordering 1", "Step 2", ord1_step2_vars))

        delta_r2, f_change, df_num, df_den, p_change = compute_f_change(model1_step1, model1_step2)
        comp_rows.append({
            "criterion_variable": criterion,
            "ordering": "Ordering 1",
            "baseline_block": "baseline predictors only",
            "added_block": "six AI readiness dimensions",
            "delta_R_squared": delta_r2,
            "F_change": f_change,
            "df_num": df_num,
            "df_den": df_den,
            "p_value_change": p_change,
        })

        coef_df1 = fit_standardized_betas(df, criterion, ord1_step2_vars)
        if len(coef_df1) > 0:
            coef_df1["criterion_variable"] = criterion
            coef_df1["ordering"] = "Ordering 1"
            coef_df1["full_model_predictors"] = ", ".join(ord1_step2_vars)
            coef_rows.append(coef_df1)

        ord2_step1_vars = readiness_factors.copy()
        ord2_step2_vars = readiness_factors + baseline_predictors

        model2_step1, sub3 = fit_ols_model(df, criterion, ord2_step1_vars)
        model2_step2, sub4 = fit_ols_model(df, criterion, ord2_step2_vars)

        step_rows.append(extract_model_step_row(model2_step1, sub3, criterion, "Ordering 2", "Step 1", ord2_step1_vars))
        step_rows.append(extract_model_step_row(model2_step2, sub4, criterion, "Ordering 2", "Step 2", ord2_step2_vars))

        delta_r2, f_change, df_num, df_den, p_change = compute_f_change(model2_step1, model2_step2)
        comp_rows.append({
            "criterion_variable": criterion,
            "ordering": "Ordering 2",
            "baseline_block": "six AI readiness dimensions only",
            "added_block": "baseline predictors",
            "delta_R_squared": delta_r2,
            "F_change": f_change,
            "df_num": df_num,
            "df_den": df_den,
            "p_value_change": p_change,
        })

        coef_df2 = fit_standardized_betas(df, criterion, ord2_step2_vars)
        if len(coef_df2) > 0:
            coef_df2["criterion_variable"] = criterion
            coef_df2["ordering"] = "Ordering 2"
            coef_df2["full_model_predictors"] = ", ".join(ord2_step2_vars)
            coef_rows.append(coef_df2)

    step_df = pd.DataFrame(step_rows)
    comp_df = pd.DataFrame(comp_rows)
    coef_df = pd.concat(coef_rows, ignore_index=True) if len(coef_rows) > 0 else pd.DataFrame()

    return step_df, comp_df, coef_df


def score_ai_readiness_factors(df: pd.DataFrame, factor_dict: Dict[str, List[str]]) -> pd.DataFrame:
    out = df.copy()
    for factor, items in factor_dict.items():
        missing = check_missing_variables(out, items, context=f"scoring {factor}")
        usable_items = [i for i in items if i not in missing]
        if len(usable_items) == 0:
            print(f"WARNING: Could not compute {factor}; all items missing.")
            out[factor] = np.nan
            continue
        out[factor] = out[usable_items].mean(axis=1)
        print(f"Computed {factor} from items: {usable_items}")
    factor_cols = list(factor_dict.keys())
    present_factor_cols = [c for c in factor_cols if c in out.columns]
    out["AI_Readiness_Total"] = out[present_factor_cols].mean(axis=1)
    print(f"Computed AI_Readiness_Total from factors: {present_factor_cols}")
    return out


def create_summary_tables(
    conv_corr: pd.DataFrame,
    conv_match: pd.DataFrame,
    disc_corr: pd.DataFrame,
    disc_latent: pd.DataFrame,
    inc_steps: pd.DataFrame,
    inc_comp: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    conv_summary_rows = []

    if len(conv_corr) > 0:
        tmp = conv_corr.copy()
        tmp["abs_r"] = tmp["r"].abs()
        for factor in tmp["predictor"].dropna().unique():
            sub = tmp[tmp["predictor"] == factor]
            conv_summary_rows.append({
                "factor": factor,
                "n_correlations": len(sub),
                "mean_r": sub["r"].mean(),
                "mean_abs_r": sub["abs_r"].mean(),
                "max_abs_r": sub["abs_r"].max(),
                "n_p_lt_05": int((sub["p_value"] < 0.05).sum()),
                "summary_type": "zero_order_convergent",
            })

    if len(conv_match) > 0:
        for _, row in conv_match.iterrows():
            avg_r_matched = row.get("avg_r_matched", np.nan)
            mean_abs_r_val = abs(avg_r_matched) if pd.notna(avg_r_matched) else np.nan
            conv_summary_rows.append({
                "factor": row.get("factor"),
                "n_correlations": row.get("k_matched", np.nan),
                "mean_r": avg_r_matched,
                "mean_abs_r": mean_abs_r_val,
                "max_abs_r": np.nan,
                "n_p_lt_05": 1 if pd.notna(row.get("p_value_approx")) and row.get("p_value_approx") < 0.05 else 0,
                "summary_type": "matched_vs_nonmatched",
            })

    conv_summary = pd.DataFrame(conv_summary_rows)

    disc_summary_rows = []
    if len(disc_corr) > 0:
        tmp = disc_corr.copy()
        tmp["abs_r"] = tmp["r"].abs()
        for factor in tmp["predictor"].dropna().unique():
            sub = tmp[tmp["predictor"] == factor]
            disc_summary_rows.append({
                "factor": factor,
                "n_observed_discriminant_correlations": len(sub),
                "mean_abs_observed_r": sub["abs_r"].mean(),
                "max_abs_observed_r": sub["abs_r"].max(),
                "n_observed_p_lt_05": int((sub["p_value"] < 0.05).sum()),
                "n_latent_tests_supporting_discriminant_validity": np.nan,
                "summary_type": "observed_discriminant",
            })

    if len(disc_latent) > 0:
        for factor in disc_latent["ai_factor_tested"].dropna().unique():
            sub = disc_latent[disc_latent["ai_factor_tested"] == factor]
            support = ((sub["p_value_diff"] < 0.05) & pd.notna(sub["p_value_diff"])).sum()
            disc_summary_rows.append({
                "factor": factor,
                "n_observed_discriminant_correlations": np.nan,
                "mean_abs_observed_r": np.nan,
                "max_abs_observed_r": np.nan,
                "n_observed_p_lt_05": np.nan,
                "n_latent_tests_supporting_discriminant_validity": int(support),
                "summary_type": "latent_discriminant",
            })

    disc_summary = pd.DataFrame(disc_summary_rows)

    inc_summary_rows = []
    if len(inc_comp) > 0:
        for criterion in inc_comp["criterion_variable"].dropna().unique():
            sub = inc_comp[inc_comp["criterion_variable"] == criterion]
            for _, row in sub.iterrows():
                inc_summary_rows.append({
                    "criterion_variable": criterion,
                    "ordering": row.get("ordering"),
                    "delta_R_squared": row.get("delta_R_squared"),
                    "F_change": row.get("F_change"),
                    "p_value_change": row.get("p_value_change"),
                    "summary_type": "incremental_comparison",
                })

    if len(inc_steps) > 0:
        for criterion in inc_steps["criterion_variable"].dropna().unique():
            sub = inc_steps[(inc_steps["criterion_variable"] == criterion) & (inc_steps["step"] == "Step 2")]
            for _, row in sub.iterrows():
                inc_summary_rows.append({
                    "criterion_variable": criterion,
                    "ordering": row.get("model_ordering"),
                    "delta_R_squared": np.nan,
                    "F_change": row.get("F_statistic"),
                    "p_value_change": row.get("model_p_value"),
                    "summary_type": "final_model_fit",
                })

    inc_summary = pd.DataFrame(inc_summary_rows)

    overview_rows = []
    if len(conv_corr) > 0:
        overview_rows.append({
            "domain": "Convergent validity",
            "metric": "Mean absolute zero-order convergent correlation",
            "value": conv_corr["r"].abs().mean(),
        })
    if len(conv_match) > 0:
        sig_count = ((conv_match["p_value_approx"] < 0.05) & pd.notna(conv_match["p_value_approx"])).sum()
        overview_rows.append({
            "domain": "Convergent validity",
            "metric": "Factors with approx. significant matched > nonmatched comparison",
            "value": sig_count,
        })
    if len(disc_corr) > 0:
        overview_rows.append({
            "domain": "Discriminant validity",
            "metric": "Mean absolute observed discriminant correlation",
            "value": disc_corr["r"].abs().mean(),
        })
    if len(disc_latent) > 0:
        support = ((disc_latent["p_value_diff"] < 0.05) & pd.notna(disc_latent["p_value_diff"])).sum()
        overview_rows.append({
            "domain": "Discriminant validity",
            "metric": "Latent nested tests supporting discriminant validity",
            "value": support,
        })
    if len(inc_comp) > 0:
        overview_rows.append({
            "domain": "Incremental validity",
            "metric": "Mean delta R-squared across hierarchical comparisons",
            "value": inc_comp["delta_R_squared"].mean(),
        })
        overview_rows.append({
            "domain": "Incremental validity",
            "metric": "Number of hierarchical comparisons with p < .05",
            "value": int(((inc_comp["p_value_change"] < 0.05) & pd.notna(inc_comp["p_value_change"])).sum()),
        })

    overview_df = pd.DataFrame(overview_rows)
    return conv_summary, disc_summary, inc_summary, overview_df


def generate_apa_writeup(
    scored_df: pd.DataFrame,
    conv_corr: pd.DataFrame,
    conv_match: pd.DataFrame,
    disc_corr: pd.DataFrame,
    disc_latent: pd.DataFrame,
    inc_steps: pd.DataFrame,
    inc_comp: pd.DataFrame
) -> str:
    lines = []

    lines.append("Phase 3 Validation Analyses for the Six-Dimensional AI Readiness Scale")
    lines.append("")
    lines.append("Scale scoring.")
    lines.append(
        "Six AI readiness factor scores were computed by averaging the four observed items assigned to each factor: "
        "AET, CLA, ALU, RDG, WHC, and CEJ. An overall AI_Readiness_Total score was then computed as the mean of the six factor scores."
    )
    lines.append("")

    lines.append("Convergent validity.")
    if len(conv_corr) > 0:
        conv_tmp = conv_corr.copy()
        conv_tmp["abs_r"] = conv_tmp["r"].abs()
        mean_abs_r = conv_tmp["abs_r"].mean()
        mean_abs_r_str = safe_str_num(mean_abs_r)
        sig_count = int(((conv_tmp["p_value"] < 0.05) & pd.notna(conv_tmp["p_value"])).sum())
        total_count = len(conv_tmp)
        lines.append(
            f"Zero-order Pearson correlations were computed between each AI readiness dimension and the convergent validity variables. "
            f"Across these analyses, the mean absolute correlation was {mean_abs_r_str}, and {sig_count} of {total_count} correlations were statistically significant at p < .05."
        )
        strongest = conv_tmp.sort_values("abs_r", ascending=False).head(3)
        if len(strongest) > 0:
            desc = []
            for _, row in strongest.iterrows():
                r_str = safe_str_num(row.get("r"))
                p_str = safe_str_num(row.get("p_value"))
                desc.append(f"{row.get('predictor')} with {row.get('outcome')} (r = {r_str}, p = {p_str})")
            lines.append("The strongest convergent associations were observed for " + "; ".join(desc) + ".")
    else:
        lines.append("Zero-order convergent validity correlations could not be estimated.")
    lines.append("")

    if len(conv_match) > 0:
        support_count = int(((conv_match["p_value_approx"] < 0.05) & (conv_match["avg_r_matched"] > conv_match["avg_r_nonmatched"])).sum())
        total_factors = len(conv_match)
        lines.append(
            "Matched versus nonmatched convergent correlation comparisons were evaluated using a Fisher r-to-z approximation applied to averaged transformed correlations. "
            f"Matched correlations were stronger than nonmatched correlations for {support_count} of {total_factors} factors based on this approximation."
        )
        for _, row in conv_match.iterrows():
            factor = row.get("factor")
            arm = safe_str_num(row.get("avg_r_matched"))
            arn = safe_str_num(row.get("avg_r_nonmatched"))
            z_str = safe_str_num(row.get("z_approx"))
            p_str = safe_str_num(row.get("p_value_approx"))
            interpretation = row.get("interpretation")
            lines.append(
                f"For {factor}, the average matched correlation was {arm}, the average nonmatched correlation was {arn}, "
                f"and the Fisher approximation yielded z = {z_str}, p = {p_str} ({interpretation})."
            )
    else:
        lines.append("Matched versus nonmatched convergent validity comparisons could not be estimated.")
    lines.append("")

    lines.append("Discriminant validity.")
    if len(disc_corr) > 0:
        disc_tmp = disc_corr.copy()
        disc_tmp["abs_r"] = disc_tmp["r"].abs()
        mean_abs_r = disc_tmp["abs_r"].mean()
        mean_abs_r_str = safe_str_num(mean_abs_r)
        sig_count = int(((disc_tmp["p_value"] < 0.05) & pd.notna(disc_tmp["p_value"])).sum())
        total_count = len(disc_tmp)
        lines.append(
            f"Observed discriminant validity was examined via zero-order correlations between the AI readiness dimensions and theoretically dissimilar variables. "
            f"The mean absolute discriminant correlation was {mean_abs_r_str}, with {sig_count} of {total_count} correlations reaching p < .05."
        )
    else:
        lines.append("Observed discriminant validity correlations could not be estimated.")
    lines.append("")

    if len(disc_latent) > 0:
        support = ((disc_latent["p_value_diff"] < 0.05) & pd.notna(disc_latent["p_value_diff"])).sum()
        total = len(disc_latent)
        lines.append(
            f"Latent discriminant validity was further examined using nested semopy measurement models. "
            f"Across {total} baseline-versus-collapsed comparisons, {int(support)} comparisons indicated that collapsing an AI readiness factor with an alternative latent construct significantly worsened model fit, which supports discriminant validity."
        )
        sample_rows = disc_latent.head(6)
        for _, row in sample_rows.iterrows():
            factor = row.get("ai_factor_tested")
            alt = row.get("alternative_factor")
            chi2d = safe_str_num(row.get("chi2_diff"))
            dfd = safe_str_num(row.get("df_diff"))
            pdiff = safe_str_num(row.get("p_value_diff"))
            dcfi = safe_str_num(row.get("delta_CFI"))
            lines.append(
                f"For {factor} versus {alt}, the chi-square difference was {chi2d} with Δdf = {dfd}, p = {pdiff}, and ΔCFI = {dcfi}."
            )
    else:
        lines.append("Latent discriminant validity comparisons could not be estimated.")
    lines.append("")

    lines.append("Incremental validity.")
    if len(inc_comp) > 0:
        sig_count = int(((inc_comp["p_value_change"] < 0.05) & pd.notna(inc_comp["p_value_change"])).sum())
        total = len(inc_comp)
        mean_delta = inc_comp["delta_R_squared"].mean()
        mean_delta_str = safe_str_num(mean_delta)
        lines.append(
            f"Incremental validity was examined with hierarchical regression analyses using two model orderings for each criterion. "
            f"Across {total} hierarchical block comparisons, the mean change in R-squared was {mean_delta_str}, and {sig_count} comparisons were statistically significant at p < .05."
        )
        for _, row in inc_comp.iterrows():
            crit = row.get("criterion_variable")
            ordering = row.get("ordering")
            dr2 = safe_str_num(row.get("delta_R_squared"))
            fch = safe_str_num(row.get("F_change"))
            pch = safe_str_num(row.get("p_value_change"))
            lines.append(
                f"For {crit} in {ordering}, the added block produced ΔR² = {dr2}, F-change = {fch}, p = {pch}."
            )
    else:
        lines.append("Hierarchical regression analyses for incremental validity could not be estimated.")
    lines.append("")

    lines.append("Overall conclusion.")
    conv_supported = False
    disc_supported = False
    inc_supported = False

    if len(conv_corr) > 0:
        conv_supported = conv_corr["r"].abs().mean() >= 0.20 or (((conv_corr["p_value"] < 0.05) & pd.notna(conv_corr["p_value"])).sum() > 0)
    if len(disc_corr) > 0 or len(disc_latent) > 0:
        obs_ok = True
        if len(disc_corr) > 0:
            obs_ok = disc_corr["r"].abs().mean() < 0.30
        latent_ok = True
        if len(disc_latent) > 0:
            latent_ok = (((disc_latent["p_value_diff"] < 0.05) & pd.notna(disc_latent["p_value_diff"])).sum() > 0)
        disc_supported = obs_ok or latent_ok
    if len(inc_comp) > 0:
        inc_supported = (((inc_comp["p_value_change"] < 0.05) & pd.notna(inc_comp["p_value_change"])).sum() > 0)

    conclusion_parts = []
    conclusion_parts.append("The six-dimensional AI readiness scale demonstrated")
    conclusion_parts.append("evidence of convergent validity" if conv_supported else "limited evidence of convergent validity")
    conclusion_parts.append(",")
    conclusion_parts.append("evidence of discriminant validity" if disc_supported else "limited evidence of discriminant validity")
    conclusion_parts.append(", and")
    conclusion_parts.append("evidence of incremental validity." if inc_supported else "limited evidence of incremental validity.")
    lines.append(" ".join(conclusion_parts))

    return "\n".join(lines)


def save_dataframe(df: pd.DataFrame, filepath: str) -> None:
    try:
        df.to_csv(filepath, index=False)
        print(f"Saved: {filepath}")
    except Exception as e:
        print(f"WARNING: Could not save {filepath}: {e}")


def main():
    print("Starting Phase 3 validation analyses...")

    if not os.path.exists(DATA_FILE):
        print(f"ERROR: Data file not found: {DATA_FILE}")
        return

    try:
        df = pd.read_csv(DATA_FILE)
        print(f"Loaded data: {DATA_FILE} with shape {df.shape}")
    except Exception as e:
        print(f"ERROR: Could not read {DATA_FILE}: {e}")
        return

    all_expected_numeric = (
        [item for items in AI_FACTORS.values() for item in items] +
        CONVERGENT_VARS +
        DISCRIMINANT_VARS +
        [item for items in ALTERNATIVE_LATENT_FACTORS.values() for item in items] +
        BASELINE_PREDICTORS +
        CRITERION_VARS
    )
    df = ensure_numeric(df, [c for c in all_expected_numeric if c in df.columns])

    print("\nScoring AI readiness factors...")
    scored_df = score_ai_readiness_factors(df, AI_FACTORS)
    save_dataframe(scored_df, OUTPUT_FILES["scored_data"])

    readiness_factor_names = list(AI_FACTORS.keys())

    print("\nRunning convergent validity analyses...")
    conv_corr = compute_zero_order_correlations(
        df=scored_df,
        predictors=readiness_factor_names,
        outcomes=CONVERGENT_VARS,
        analysis_label="convergent_validity"
    )
    save_dataframe(conv_corr, OUTPUT_FILES["conv_corr"])

    conv_match = compare_matched_vs_nonmatched_convergent(
        df=scored_df,
        factors=readiness_factor_names,
        convergent_vars=CONVERGENT_VARS,
        matched_dict=MATCHED_CONVERGENT
    )
    save_dataframe(conv_match, OUTPUT_FILES["conv_match"])

    if len(conv_corr) > 0:
        print("\nConvergent validity correlation summary:")
        print(conv_corr.head(20).to_string(index=False))
    if len(conv_match) > 0:
        print("\nMatched vs nonmatched convergent summary:")
        print(conv_match.to_string(index=False))

    print("\nRunning discriminant validity analyses...")
    disc_corr = compute_zero_order_correlations(
        df=scored_df,
        predictors=readiness_factor_names,
        outcomes=DISCRIMINANT_VARS,
        analysis_label="discriminant_validity"
    )
    save_dataframe(disc_corr, OUTPUT_FILES["disc_corr"])

    disc_latent = run_discriminant_latent_tests(
        df=scored_df,
        ai_factors=AI_FACTORS,
        alternative_factors=ALTERNATIVE_LATENT_FACTORS
    )
    save_dataframe(disc_latent, OUTPUT_FILES["disc_latent"])

    if len(disc_corr) > 0:
        print("\nDiscriminant validity observed correlation summary:")
        print(disc_corr.head(20).to_string(index=False))
    if len(disc_latent) > 0:
        print("\nDiscriminant validity latent model comparison summary:")
        print(disc_latent.head(20).to_string(index=False))

    print("\nRunning incremental validity analyses...")
    inc_steps, inc_comp, inc_coef = run_incremental_validity(
        df=scored_df,
        readiness_factors=readiness_factor_names,
        baseline_predictors=BASELINE_PREDICTORS,
        criterion_vars=CRITERION_VARS
    )
    save_dataframe(inc_steps, OUTPUT_FILES["inc_steps"])
    save_dataframe(inc_comp, OUTPUT_FILES["inc_comp"])
    save_dataframe(inc_coef, OUTPUT_FILES["inc_coef"])

    if len(inc_steps) > 0:
        print("\nIncremental validity model steps summary:")
        print(inc_steps.head(20).to_string(index=False))
    if len(inc_comp) > 0:
        print("\nIncremental validity model comparison summary:")
        print(inc_comp.head(20).to_string(index=False))
    if len(inc_coef) > 0:
        print("\nIncremental validity full-model coefficients summary:")
        print(inc_coef.head(20).to_string(index=False))

    print("\nCreating summary tables...")
    conv_summary, disc_summary, inc_summary, overview_df = create_summary_tables(
        conv_corr=conv_corr,
        conv_match=conv_match,
        disc_corr=disc_corr,
        disc_latent=disc_latent,
        inc_steps=inc_steps,
        inc_comp=inc_comp
    )

    save_dataframe(conv_summary, OUTPUT_FILES["conv_summary"])
    save_dataframe(disc_summary, OUTPUT_FILES["disc_summary"])
    save_dataframe(inc_summary, OUTPUT_FILES["inc_summary"])
    save_dataframe(overview_df, OUTPUT_FILES["all_overview"])

    print("\nGenerating APA-style write-up...")
    apa_text = generate_apa_writeup(
        scored_df=scored_df,
        conv_corr=conv_corr,
        conv_match=conv_match,
        disc_corr=disc_corr,
        disc_latent=disc_latent,
        inc_steps=inc_steps,
        inc_comp=inc_comp
    )
    try:
        with open(OUTPUT_FILES["apa_writeup"], "w", encoding="utf-8") as f:
            f.write(apa_text)
        print(f"Saved: {OUTPUT_FILES['apa_writeup']}")
    except Exception as e:
        print(f"WARNING: Could not save APA write-up: {e}")

    print("\nPhase 3 validation analyses complete.")
    print("\nGenerated files:")
    for key, path in OUTPUT_FILES.items():
        exists_flag = os.path.exists(path)
        print(f" - {path}: {'OK' if exists_flag else 'NOT FOUND'}")


if __name__ == "__main__":
    main()

Starting Phase 3 validation analyses...
Loaded data: ai_readiness_validity_data.csv with shape (400, 61)

Scoring AI readiness factors...
Computed AET from items: ['AET1', 'AET2', 'AET3', 'AET4']
Computed CLA from items: ['CLA1', 'CLA2', 'CLA3', 'CLA4']
Computed ALU from items: ['ALU1', 'ALU2', 'ALU3', 'ALU4']
Computed RDG from items: ['RDG1', 'RDG2', 'RDG3', 'RDG4']
Computed WHC from items: ['WHC1', 'WHC2', 'WHC3', 'WHC4']
Computed CEJ from items: ['CEJ1', 'CEJ2', 'CEJ3', 'CEJ4']
Computed AI_Readiness_Total from factors: ['AET', 'CLA', 'ALU', 'RDG', 'WHC', 'CEJ']
Saved: ai_readiness_scored_data.csv

Running convergent validity analyses...
Saved: convergent_validity_correlations.csv
Saved: convergent_validity_matched_vs_nonmatched.csv

Convergent validity correlation summary:
           analysis predictor                       outcome   n        r      p_value effect_size_label
convergent_validity       AET              ai_self_efficacy 400 0.446037 5.976311e-21            medium
conve

Phase 4 - Criterion-Related Validity

In [ ]:
# ============================================================
# APA-Style Descriptive Statistics, Correlations, and Write-Up
# Dataset: ai_readiness_criterion_validation_fake_dataset.csv
#
# This script creates:
#   1. APA-style descriptive statistics and correlations table
#   2. APA-style written results summary
#
# Outputs:
#   1. apa_descriptives_correlations_table.txt
#   2. apa_descriptives_correlations_table.csv
#   3. apa_descriptives_correlations_table.xlsx
#   4. apa_correlation_results_writeup.txt
#
# Required packages:
#   pandas
#   numpy
#   scipy
#   openpyxl
# ============================================================

import pandas as pd
import numpy as np
from scipy import stats
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ============================================================
# 1. User settings
# ============================================================

DATA_FILE = "ai_readiness_criterion_validation_data.csv"

DIMENSION_VARS = [
    "AET_score",
    "CLA_score",
    "ALU_score",
    "RDG_score",
    "WHC_score",
    "CEJ_score"
]

OUTCOME_VARS = [
    "Supervisor_Performance",
    "Employee_Engagement"
]

OVERALL_VAR = "Overall_AI_Readiness"

REPORTING_LABELS = {
    "AET_score": "AET",
    "CLA_score": "CLA",
    "ALU_score": "ALU",
    "RDG_score": "RDG",
    "WHC_score": "WHC",
    "CEJ_score": "CEJ",
    "Overall_AI_Readiness": "Overall AI readiness",
    "Supervisor_Performance": "Supervisor-rated job performance",
    "Employee_Engagement": "Employee engagement"
}

# Optional:
# Add reliability estimates here if you want values on the diagonal.
# If a variable is not listed, its diagonal cell will remain blank.
#
# Example:
# SCALE_RELIABILITIES = {
#     "AET_score": .88,
#     "CLA_score": .84,
#     "ALU_score": .79,
#     "RDG_score": .81,
#     "WHC_score": .86,
#     "CEJ_score": .83,
#     "Overall_AI_Readiness": .91
# }
SCALE_RELIABILITIES = {}


# ============================================================
# 2. Helper functions
# ============================================================

def significance_stars(p):
    """
    Return significance stars for p-values.
    """
    if pd.isna(p):
        return ""
    elif p < .001:
        return "***"
    elif p < .01:
        return "**"
    elif p < .05:
        return "*"
    else:
        return ""


def format_number(value, decimals=2, remove_leading_zero=True):
    """
    Format numbers APA-style.

    Examples:
        0.72 becomes .72
        -0.32 becomes -.32
        3.75 stays 3.75
    """
    if pd.isna(value):
        return ""

    formatted = f"{value:.{decimals}f}"

    if remove_leading_zero:
        if formatted.startswith("0."):
            formatted = formatted.replace("0.", ".", 1)
        elif formatted.startswith("-0."):
            formatted = formatted.replace("-0.", "-.", 1)

    return formatted


def format_p_value(p):
    """
    Format p-values APA-style.
    """
    if pd.isna(p):
        return ""

    if p < .001:
        return "p < .001"
    else:
        return f"p = {format_number(p, decimals=3)}"


def pearson_pairwise(df, var1, var2):
    """
    Compute Pearson correlation using pairwise complete cases.
    """
    temp = df[[var1, var2]].dropna()

    if len(temp) < 3:
        return np.nan, np.nan, len(temp)

    r, p = stats.pearsonr(temp[var1], temp[var2])
    return r, p, len(temp)


def format_correlation(r, p):
    """
    Format a Pearson correlation with significance stars.
    """
    if pd.isna(r):
        return ""

    return f"{format_number(r, decimals=2)}{significance_stars(p)}"


def create_apa_correlation_table(df, variables):
    """
    Create APA-style descriptives and lower-triangle correlation table.
    """
    rows = []

    for i, var_i in enumerate(variables):
        row = {
            "Variable": f"{i + 1}. {REPORTING_LABELS[var_i]}",
            "Mean": format_number(df[var_i].mean(), decimals=2),
            "SD": format_number(df[var_i].std(ddof=1), decimals=2)
        }

        for j, var_j in enumerate(variables):
            col_name = str(j + 1)

            if j < i:
                r, p, n = pearson_pairwise(df, var_i, var_j)
                row[col_name] = format_correlation(r, p)

            elif j == i:
                if var_i in SCALE_RELIABILITIES:
                    row[col_name] = format_number(
                        SCALE_RELIABILITIES[var_i],
                        decimals=2
                    )
                else:
                    row[col_name] = ""

            else:
                row[col_name] = ""

        rows.append(row)

    return pd.DataFrame(rows)


def create_console_table(table, title, subtitle, sample_n):
    """
    Create a formatted text table that resembles a published APA table.
    """
    lines = []

    variables_count = len([col for col in table.columns if col.isdigit()])

    col_widths = {
        "Variable": 42,
        "Mean": 8,
        "SD": 8
    }

    for i in range(1, variables_count + 1):
        col_widths[str(i)] = 8

    total_width = (
        col_widths["Variable"]
        + col_widths["Mean"]
        + col_widths["SD"]
        + sum(col_widths[str(i)] for i in range(1, variables_count + 1))
    )

    lines.append(title.center(total_width))
    lines.append(subtitle.center(total_width))
    lines.append("=" * total_width)

    header = (
        " " * col_widths["Variable"]
        + "Mean".center(col_widths["Mean"])
        + "SD".center(col_widths["SD"])
    )

    for i in range(1, variables_count + 1):
        header += str(i).center(col_widths[str(i)])

    lines.append(header)
    lines.append("-" * total_width)

    lines.append(f"AI-readiness dimensions (N = {sample_n})")

    dimension_rows = table.iloc[0:6]
    overall_row = table.iloc[6:7]
    outcome_rows = table.iloc[7:]

    for _, row in dimension_rows.iterrows():
        line = str(row["Variable"]).ljust(col_widths["Variable"])
        line += str(row["Mean"]).center(col_widths["Mean"])
        line += str(row["SD"]).center(col_widths["SD"])

        for i in range(1, variables_count + 1):
            line += str(row[str(i)]).center(col_widths[str(i)])

        lines.append(line)

    lines.append("")
    lines.append(f"Composite score (N = {sample_n})")

    for _, row in overall_row.iterrows():
        line = str(row["Variable"]).ljust(col_widths["Variable"])
        line += str(row["Mean"]).center(col_widths["Mean"])
        line += str(row["SD"]).center(col_widths["SD"])

        for i in range(1, variables_count + 1):
            line += str(row[str(i)]).center(col_widths[str(i)])

        lines.append(line)

    lines.append("")
    lines.append(f"Criterion outcomes (N = {sample_n})")

    for _, row in outcome_rows.iterrows():
        line = str(row["Variable"]).ljust(col_widths["Variable"])
        line += str(row["Mean"]).center(col_widths["Mean"])
        line += str(row["SD"]).center(col_widths["SD"])

        for i in range(1, variables_count + 1):
            line += str(row[str(i)]).center(col_widths[str(i)])

        lines.append(line)

    lines.append("=" * total_width)
    lines.append(
        "Note. Values below the diagonal are Pearson correlations. "
        "Correlations are based on pairwise complete cases. "
        "Values on the diagonal are reliability estimates when provided."
    )
    lines.append("***p < .001. **p < .01. *p < .05.")

    return "\n".join(lines)


def save_excel_table(table, file_name):
    """
    Save the APA-style table to Excel with light formatting.
    """
    table.to_excel(file_name, index=False)

    wb = load_workbook(file_name)
    ws = wb.active
    ws.title = "APA Correlation Table"

    thin = Side(border_style="thin", color="000000")
    thick = Side(border_style="medium", color="000000")

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")
        cell.border = Border(top=thick, bottom=thin)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top")

    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)

        for cell in col:
            cell_value = "" if cell.value is None else str(cell.value)
            max_length = max(max_length, len(cell_value))

        ws.column_dimensions[col_letter].width = min(max_length + 2, 45)

    wb.save(file_name)


def get_correlation_result(df, var1, var2):
    """
    Return a dictionary containing correlation details for two variables.
    """
    r, p, n = pearson_pairwise(df, var1, var2)

    return {
        "var1": var1,
        "var2": var2,
        "label1": REPORTING_LABELS[var1],
        "label2": REPORTING_LABELS[var2],
        "r": r,
        "p": p,
        "n": n,
        "significant": p < .05 if not pd.isna(p) else False
    }


def summarize_correlation_range(correlation_results):
    """
    Summarize the range of correlations from a list of correlation result dictionaries.
    """
    valid_results = [
        result for result in correlation_results
        if not pd.isna(result["r"])
    ]

    if len(valid_results) == 0:
        return "No valid correlations could be computed."

    rs = [result["r"] for result in valid_results]
    min_r = min(rs)
    max_r = max(rs)

    return f"r values ranged from {format_number(min_r, decimals=2)} to {format_number(max_r, decimals=2)}"


def summarize_significant_dimension_correlations(df, outcome_var, dimension_vars):
    """
    Summarize which AI-readiness dimensions were significantly associated
    with a specified outcome.
    """
    results = [
        get_correlation_result(df, dim, outcome_var)
        for dim in dimension_vars
    ]

    significant_results = [
        result for result in results
        if result["significant"]
    ]

    outcome_label = REPORTING_LABELS[outcome_var]

    if len(significant_results) == 0:
        return (
            f"None of the six AI-readiness dimensions were significantly "
            f"associated with {outcome_label}."
        )

    parts = []

    for result in significant_results:
        parts.append(
            f"{result['label1']} "
            f"({format_number(result['r'], decimals=2)}, {format_p_value(result['p'])})"
        )

    joined_parts = "; ".join(parts)

    return (
        f"The AI-readiness dimensions significantly associated with "
        f"{outcome_label} were: {joined_parts}."
    )


def create_apa_results_writeup(df, dimension_vars, overall_var, outcome_vars):
    """
    Create an APA-style written summary of the descriptive statistics
    and correlations.
    """
    total_n = len(df)

    # Correlations among the six AI-readiness dimensions
    dimension_intercorrelations = []

    for i, var_i in enumerate(dimension_vars):
        for j, var_j in enumerate(dimension_vars):
            if j < i:
                dimension_intercorrelations.append(
                    get_correlation_result(df, var_i, var_j)
                )

    # Correlations between overall AI readiness and the six dimensions
    overall_dimension_correlations = [
        get_correlation_result(df, overall_var, dim)
        for dim in dimension_vars
    ]

    # Correlations between overall AI readiness and the two criterion outcomes
    overall_outcome_correlations = [
        get_correlation_result(df, overall_var, outcome)
        for outcome in outcome_vars
    ]

    supervisor_var = "Supervisor_Performance"
    engagement_var = "Employee_Engagement"

    supervisor_summary = summarize_significant_dimension_correlations(
        df=df,
        outcome_var=supervisor_var,
        dimension_vars=dimension_vars
    )

    engagement_summary = summarize_significant_dimension_correlations(
        df=df,
        outcome_var=engagement_var,
        dimension_vars=dimension_vars
    )

    overall_outcome_parts = []

    for result in overall_outcome_correlations:
        overall_outcome_parts.append(
            f"{result['label2']} "
            f"({format_number(result['r'], decimals=2)}, {format_p_value(result['p'])})"
        )

    overall_outcome_sentence = "; ".join(overall_outcome_parts)

    writeup = f"""
APA-Style Results Summary

Means, standard deviations, and Pearson correlations were computed for the six AI-readiness dimensions, overall AI readiness, and the two criterion outcomes using a sample of {total_n} employees. Correlations were estimated using pairwise complete cases to retain all available data for each bivariate association.

The six AI-readiness dimensions were positively intercorrelated, with {summarize_correlation_range(dimension_intercorrelations)}. Overall AI readiness was positively associated with each of the six AI-readiness dimensions, with {summarize_correlation_range(overall_dimension_correlations)}. Overall AI readiness was also correlated with the criterion outcomes as follows: {overall_outcome_sentence}.

{supervisor_summary}

{engagement_summary}

Taken together, the pattern of correlations provides preliminary criterion-related validity evidence for the AI-readiness scale. Specifically, the results suggest that individual differences in AI readiness are meaningfully associated with supervisor-rated job performance and employee engagement.
"""

    return writeup.strip()


# ============================================================
# 3. Load data
# ============================================================

df = pd.read_csv(DATA_FILE)

required_vars = DIMENSION_VARS + OUTCOME_VARS
missing_vars = [var for var in required_vars if var not in df.columns]

if missing_vars:
    raise ValueError(
        "The following required variables are missing from the dataset: "
        + ", ".join(missing_vars)
    )


# ============================================================
# 4. Create overall AI-readiness score
# ============================================================

df[OVERALL_VAR] = df[DIMENSION_VARS].mean(axis=1, skipna=True)


# ============================================================
# 5. Create APA-style table
# ============================================================

TABLE_VARS = DIMENSION_VARS + [OVERALL_VAR] + OUTCOME_VARS

apa_table = create_apa_correlation_table(
    df=df,
    variables=TABLE_VARS
)

sample_n = len(df)

console_output = create_console_table(
    table=apa_table,
    title="TABLE 1",
    subtitle="Descriptive Statistics and Correlations for AI-Readiness Criterion Validation",
    sample_n=sample_n
)


# ============================================================
# 6. Create APA-style results write-up
# ============================================================

apa_writeup = create_apa_results_writeup(
    df=df,
    dimension_vars=DIMENSION_VARS,
    overall_var=OVERALL_VAR,
    outcome_vars=OUTCOME_VARS
)


# ============================================================
# 7. Save outputs
# ============================================================

apa_table.to_csv("apa_descriptives_correlations_table.csv", index=False)

with open("apa_descriptives_correlations_table.txt", "w", encoding="utf-8") as f:
    f.write(console_output)

with open("apa_correlation_results_writeup.txt", "w", encoding="utf-8") as f:
    f.write(apa_writeup)

save_excel_table(
    table=apa_table,
    file_name="apa_descriptives_correlations_table.xlsx"
)


# ============================================================
# 8. Print outputs to console
# ============================================================

print("\n")
print(console_output)

print("\n")
print("=" * 100)
print("APA-STYLE RESULTS WRITE-UP")
print("=" * 100)
print(apa_writeup)

print("\nFiles saved:")
print("apa_descriptives_correlations_table.txt")
print("apa_descriptives_correlations_table.csv")
print("apa_descriptives_correlations_table.xlsx")
print("apa_correlation_results_writeup.txt")



                                                             TABLE 1                                                              
                          Descriptive Statistics and Correlations for AI-Readiness Criterion Validation                           
                                            Mean     SD      1       2       3       4       5       6       7       8       9    
----------------------------------------------------------------------------------------------------------------------------------
AI-readiness dimensions (N = 350)
1. AET                                      3.24    .94                                                                           
2. CLA                                      3.24    .95     .03                                                                   
3. ALU                                      3.24    .92     .01     -.03                                                          
4. RDG                                      3.2

The End!!
